# 1 - Downloading

In [1]:
import os

print("--- ⬇️ TÉLÉCHARGEMENT DE IMAGEWOOF ---")
%cd /content/

# Téléchargement ultra-rapide depuis les serveurs AWS de FastAI
!wget https://s3.amazonaws.com/fast-ai-imageclas/imagewoof2.tgz

# Décompression silencieuse
print("⏳ Décompression de l'archive...")
!tar -xzf imagewoof2.tgz

print("✅ Imagewoof est prêt sur le SSD local (/content/imagewoof2/)")

--- ⬇️ TÉLÉCHARGEMENT DE IMAGEWOOF ---
/content
--2026-04-24 10:26:08--  https://s3.amazonaws.com/fast-ai-imageclas/imagewoof2.tgz
Resolving s3.amazonaws.com (s3.amazonaws.com)... 16.15.244.22, 52.217.225.136, 52.217.134.160, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|16.15.244.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1343715595 (1.3G) [application/x-tar]
Saving to: ‘imagewoof2.tgz’

imagewoof2.tgz      100%[===================>]   1.25G  39.3MB/s    in 36s     

2026-04-24 10:26:44 (35.6 MB/s) - ‘imagewoof2.tgz’ saved [1343715595/1343715595]

⏳ Décompression de l'archive...
✅ Imagewoof est prêt sur le SSD local (/content/imagewoof2/)


In [ ]:
import os
import shutil
import random

print("--- CREATION DU DATASET MULTI-FIDELITE (Imagewoof) ---")
print("Les images BF sont stockees PROPRES : la degradation BF (downsample + bruit")
print("+ JPEG) est appliquee A LA VOLEE au train et au test via src/degradation.py.")

# Chemins
raw_dir = "/content/imagewoof2"
out_dir = "/content/processed_multifidelity"

# On nettoie si le dossier existe deja
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)

# Creation de l'arborescence
os.makedirs(f"{out_dir}/test", exist_ok=True)
os.makedirs(f"{out_dir}/train/HF", exist_ok=True)
os.makedirs(f"{out_dir}/train/BF", exist_ok=True)

classes = os.listdir(f"{raw_dir}/train")

print("Copie des images (aucune degradation sur disque)...")

for cls in classes:
    # 1. Copie integrale du dossier de validation vers "test" (100% HF propre)
    shutil.copytree(f"{raw_dir}/val/{cls}", f"{out_dir}/test/{cls}")

    # 2. Traitement du dossier d'entrainement
    os.makedirs(f"{out_dir}/train/HF/{cls}", exist_ok=True)
    os.makedirs(f"{out_dir}/train/BF/{cls}", exist_ok=True)

    images = os.listdir(f"{raw_dir}/train/{cls}")
    random.seed(42)  # Pour etre reproductible (meme split qu'avant)
    random.shuffle(images)

    # Split : 10% HF, 90% BF (identique a l'ancien pipeline)
    split_idx = int(len(images) * 0.10)
    hf_images = images[:split_idx]
    bf_images = images[split_idx:]

    # HF et BF sont tous deux copies PROPRES (la degradation BF est a la volee)
    for img_name in hf_images:
        shutil.copy2(f"{raw_dir}/train/{cls}/{img_name}", f"{out_dir}/train/HF/{cls}/{img_name}")
    for img_name in bf_images:
        shutil.copy2(f"{raw_dir}/train/{cls}/{img_name}", f"{out_dir}/train/BF/{cls}/{img_name}")

print("Termine ! Dataset multi-fidelite (images propres) pret sur le SSD.")

In [3]:
from google.colab import drive
drive.mount('/content/drive')

print("--- 📦 ZIPPAGE ET SAUVEGARDE SUR DRIVE ---")

# 1. On zip le dossier traité en utilisant Linux (Ultra Rapide)
print("⏳ Zippage du dataset au niveau 0 (aucune compression, juste un conteneur)...")
!cd /content/ && zip -0 -r -q imagewoof_multifidelity.zip processed_multifidelity/

# 2. On l'envoie sur ton Drive
drive_path = "/content/drive/MyDrive/UTBM_PF22/datasets/Imagewoof"
os.makedirs(drive_path, exist_ok=True)

print("⏳ Transfert du fichier ZIP vers ton Google Drive...")
import shutil
shutil.copy2("/content/imagewoof_multifidelity.zip", f"{drive_path}/dataset_multifidelity.zip")

print(f"✅ TOUT EST BON ! Ton nouveau dataset est sauvegardé et sécurisé dans : {drive_path}")

Mounted at /content/drive
--- 📦 ZIPPAGE ET SAUVEGARDE SUR DRIVE ---
⏳ Zippage du dataset au niveau 0 (aucune compression, juste un conteneur)...
⏳ Transfert du fichier ZIP vers ton Google Drive...
✅ TOUT EST BON ! Ton nouveau dataset est sauvegardé et sécurisé dans : /content/drive/MyDrive/UTBM_PF22/datasets/Imagewoof
